In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import openpyxl
import pandas as pd
from scipy import stats

energy_mix = pd.ExcelFile("DOE-538281345095_Energy_Mix.xlsx")
electricity_rates = pd.ExcelFile("DOE-538281345095_Electricity_Rates.xlsx")
customer_sales = pd.ExcelFile("DOE-538281345095_Customer_Sales_and_Count_20_25.xlsx")

#check the tabs in excel
print(f"Energy mix: {energy_mix.sheet_names}")
print(f"Electricty rates: {electricity_rates.sheet_names}")
print(f"Customer Sales: {customer_sales.sheet_names}")

<h2 style="color:red; font-weight:bold">Process the Electric rates

In [ ]:
residential_nonlifeline_rates = pd.read_excel("DOE-538281345095_Electricity_Rates.xlsx", sheet_name="Residential Non-Lifeline")
# for residential areas that do not qualify for low-income assistance
# evaluates to NaN
residential_nlrates = pd.read_excel("DOE-538281345095_Electricity_Rates.xlsx", sheet_name="NLResidential Rates") # lifeline(?)
commercial_rates = pd.read_excel("DOE-538281345095_Electricity_Rates.xlsx", sheet_name="Commercial Rates")


#print(residential_nonlifeline_rates.head(), residential_nonlifeline_rates.shape)
print(f"Residential: {residential_nlrates.head()},"
      f" Shape: {residential_nlrates.shape},"
      f" Info: {residential_nlrates.info()},"
      f"Describe: {residential_nlrates.describe(include="all")},"
      f"Number of null: {residential_nlrates.isna().sum()}") # Shape (147, 78), 0 NaN
print(f"Commercial: {commercial_rates.head()}, Shape: {commercial_rates.shape}") # Shape ( 152, 78 )


In [ ]:
#Clean Data
res = residential_nlrates.copy()
res.columns = res.iloc[0]
res = res.iloc[1:].reset_index(drop=True)

com = commercial_rates.copy()
com.columns = com.iloc[0]
com = com.iloc[1:].reset_index(drop=True)
print(com.columns.tolist())
print(res.columns.tolist())

In [ ]:
id_cols = ["Distribution Utility", "ShortHand", "GRID", "Region", "Island", "EC/PDU"]
month_cols = [
    col for col in res.columns
    if str(col).endswith("-20") or str(col).endswith("-24")  or str(col).endswith("-21")  or str(col).endswith("-25")
]
print(month_cols)


In [ ]:
processed_cols_res = res.melt(
    id_vars= id_cols,
    value_vars = month_cols,
    var_name="Date",
    value_name="Rate",
)

processed_cols_com = com.melt(
    id_vars= id_cols,
    value_vars = month_cols,
    var_name="Date",
    value_name="Rate",
)

processed_cols_res["Customer Class"] = "Residential"
processed_cols_com["Customer Class"] = "Commercial"

rates = pd.concat(
    [processed_cols_res, processed_cols_com],
    ignore_index=True
)

rates["Rate"] = pd.to_numeric(
    rates["Rate"],
    errors="coerce"
)

rates["Date"] = pd.to_datetime(
    rates["Date"],
    format="%b-%y",
    errors="coerce"
)

rates_luzon = rates[
    rates["Island"].astype(str).str.strip().str.lower() == "luzon"
].copy()
print(rates_luzon) #final electricity rates data processed



<h2 style="color:red; font-weight:bold">Process Customer sales shit

In [ ]:
raw = pd.read_excel("DOE-538281345095_Customer_Sales_and_Count_20_25.xlsx",  sheet_name="Sales and Customer Count",
    header=None)

print(raw.shape)

years = [2020, 2021, 2024, 2025]

customer_classes = [
    "Residential",
    "Commercial",
    "Industrial",
    "Others"
]

records = []

for year in years:

    sales_title = f"Customer Sales as of {year}"
    sales_title_rows = raw.index[
        raw.iloc[:, 1].astype(str).str.strip() == sales_title
    ]

    customer_title = f"Number of Customer as of {year}"
    customer_title_rows = raw.index[
        raw.iloc[:, 1].astype(str).str.strip() == customer_title
    ]

    if len(sales_title_rows) == 0:
        print(f"Sales section not found for {year}")
        continue

    if len(customer_title_rows) == 0:
        print(f"Customer section not found for {year}")
        continue

    sales_title_row = sales_title_rows[0]
    customer_title_row = customer_title_rows[0]

    sales_start = sales_title_row + 2

    customer_start = customer_title_row + 2

    for i, customer_class in enumerate(customer_classes):

        sales_row = sales_start + i
        customer_row = customer_start + i

        for month_num in range(1, 13):

            sales_value = raw.iloc[sales_row, month_num + 1]
            customer_value = raw.iloc[customer_row, month_num + 1]

            records.append({
                "Year": year,
                "Month": month_num,
                "Customer Class": customer_class,
                "Sales": pd.to_numeric(
                    sales_value,
                    errors="coerce"
                ),
                "Customer Count": pd.to_numeric(
                    customer_value,
                    errors="coerce"
                )
            })

sales_count_clean = pd.DataFrame(records)

print(sales_count_clean.head(20))
print(sales_count_clean.shape)


In [ ]:
sales_count_clean["Date"] = pd.to_datetime(
    sales_count_clean["Year"].astype(str)
    + "-"
    + sales_count_clean["Month"].astype(str)
    + "-01"
)
# Sort
sales_count_final = sales_count_clean.sort_values(
    ["Date", "Customer Class"]
).reset_index(drop=True)

print(sales_count_final.head())
print(sales_count_final.tail())
print(sales_count_final["Year"].value_counts().sort_index())



In [ ]:
philippine_sales = sales_count_final[
    sales_count_final["Customer Class"].isin(
        ["Residential", "Commercial"]
    )
].copy()
print(philippine_sales["Customer Class"].value_counts())
#final sales is philippine_sales
annual_sales = (
    philippine_sales
    .groupby(["Year", "Customer Class"])["Sales"]
    .sum()
    .reset_index()
)

print(annual_sales)

<h2 style="color:red; font-weight:bold">Process Energy Mix

In [ ]:
energy_file = pd.read_excel("DOE-538281345095_Energy_Mix.xlsx", header=None, sheet_name="Power Generation Mix")

print(energy_file.shape)
print(energy_file.head(20))

In [ ]:
for i in range(len(energy_file)):
    row_text = " ".join(
        map(str, energy_file.iloc[i].tolist())
    )

    if "LUZON" in row_text.upper():
        print(i, row_text)

In [ ]:
years = {
    "2020": 18,
    "2021": 19,
    "2024": 22,
    "2025": 23
}

top_level_sources = [
    "Coal",
    "Oil-based",
    "Natural Gas",
    "Renewable Energy (RE)"
]

energy_mix_luzon = energy_file.iloc[7:15, [0, 18, 19, 22, 23]].copy()

energy_mix_luzon.columns = [
    "Energy Source",
    "2020",
    "2021",
    "2024",
    "2025"
]

energy_mix_luzon["Energy Source"] = (
    energy_mix_luzon["Energy Source"]
    .astype(str)
    .str.strip()
)

for year in ["2020", "2021", "2024", "2025"]:
    energy_mix_luzon[year] = pd.to_numeric(
        energy_mix_luzon[year],
        errors="coerce"
    )

energy_mix_luzon = energy_mix_luzon[
    energy_mix_luzon["Energy Source"].isin(top_level_sources)
].copy()

print(energy_mix_luzon.to_string(index=False))

In [ ]:
total_generation = {
    "2020": 72418709,
    "2021": 75242541,
    "2024": 90269376,
    "2025": 85945490.35
}

for year in ["2020", "2021", "2024", "2025"]:
    energy_mix_luzon[f"Share {year} (%)"] = (
        energy_mix_luzon[year]
        / total_generation[year]
        * 100
    )

print(
    energy_mix_luzon.round(3).to_string(index=False)
)

In [ ]:
energy_mix_final = energy_mix_luzon.melt(
    id_vars="Energy Source",
    value_vars=[
        "Share 2020 (%)",
        "Share 2021 (%)",
        "Share 2024 (%)",
        "Share 2025 (%)"
    ],
    var_name="Year",
    value_name="Share"
)

energy_mix_final["Year"] = (
    energy_mix_final["Year"]
    .str.extract(r"(\d{4})")
    .astype(int)
)

print(energy_mix_final.to_string(index=False))

### HOURLY DEMAND
:cry

In [ ]:
hourly_file = pd.read_excel("Hourly-Demand.xlsx", header=None)

headers = hourly_file.iloc[1].copy()

headers.iloc[0] = "Date"

headers = headers.iloc[:25]

hourly_wide = hourly_file.iloc[2:, :25].copy()

hourly_wide.columns = headers

print(hourly_wide, headers)


In [ ]:
hourly_wide["Date"] = pd.to_datetime(
    hourly_wide["Date"],
    errors="coerce"
)

hourly_wide = hourly_wide.dropna(
    subset=["Date"]
).copy()

hour_columns = list(range(1, 25))

for hour in hour_columns:
    hourly_wide[hour] = pd.to_numeric(
        hourly_wide[hour],
        errors="coerce"
    )

hourly_wide["Year"] = hourly_wide["Date"].dt.year
hourly_wide["Month"] = hourly_wide["Date"].dt.month
hourly_wide["Day"] = hourly_wide["Date"].dt.day
hourly_wide["Day_of_Week"] = (
    hourly_wide["Date"].dt.day_name()
)

study_years = [
    2020,
    2021,
    2024,
    2025
]

hourly_wide = hourly_wide[
    hourly_wide["Year"].isin(study_years)
].copy()


In [ ]:
hourly_demand = hourly_wide.melt(
    id_vars=[
        "Date",
        "Year",
        "Month",
        "Day",
        "Day_of_Week"
    ],
    value_vars=hour_columns,
    var_name="Hour",
    value_name="Demand_MW"
)


hourly_demand["Hour"] = pd.to_numeric(
    hourly_demand["Hour"],
    errors="coerce"
)

hourly_demand["Demand_MW"] = pd.to_numeric(
    hourly_demand["Demand_MW"],
    errors="coerce"
)

hourly_demand = hourly_demand.dropna(
    subset=["Hour", "Demand_MW"]
).copy()


print(
    "Processed hourly observations:",
    len(hourly_demand)
)

display(hourly_demand.head(24))

<h2 style="color:red; font-weight:bold">Comparison HERE</h2>

In [ ]:

hourly_shape_year = (
    hourly_demand
    .groupby(
        ["Year", "Hour"],
        as_index=False
    )
    .agg(
        Average_Demand_MW=("Demand_MW", "mean"),
        Maximum_Demand_MW=("Demand_MW", "max"),
        Minimum_Demand_MW=("Demand_MW", "min")
    )
)

display(hourly_shape_year)

In [ ]:
peak_idx = (
    hourly_demand
    .groupby("Year")["Demand_MW"]
    .idxmax()
)

annual_peak = (
    hourly_demand
    .loc[peak_idx]
    .sort_values("Year")
    [
        [
            "Year",
            "Date",
            "Hour",
            "Demand_MW"
        ]
    ]
    .rename(
        columns={
            "Demand_MW": "Peak_Demand_MW"
        }
    )
    .reset_index(drop=True)
)

display(annual_peak)

In [ ]:
#energy_mix_final
#philippine_sales
#rates_luzon
WFH_YEARS = [2020, 2021]
OFFICE_YEARS = [2024, 2025]
CUSTOMER_CLASSES = ["Residential", "Commercial"]


print(energy_mix_final.head(20))
print(philippine_sales)
print(rates_luzon)

In [ ]:
hourly_demand["Work_Mode"] = "Other"

hourly_demand.loc[
    hourly_demand["Year"].isin(WFH_YEARS),
    "Work_Mode"
] = "WFH"

hourly_demand.loc[
    hourly_demand["Year"].isin(OFFICE_YEARS),
    "Work_Mode"
] = "Office"


# Keep only the two study scenarios
hourly_demand = hourly_demand[
    hourly_demand["Work_Mode"].isin(["WFH", "Office"])
].copy()


display(
    hourly_demand[
        ["Date", "Year", "Hour", "Demand_MW", "Work_Mode"]
    ].head()
)

In [ ]:

work_mode_hourly = (
    hourly_demand
    .groupby(
        ["Work_Mode", "Hour"],
        as_index=False
    )
    .agg(
        Average_Demand_MW=("Demand_MW", "mean"),
        Maximum_Demand_MW=("Demand_MW", "max"),
        Minimum_Demand_MW=("Demand_MW", "min")
    )
)

display(work_mode_hourly)

In [ ]:
#hypothetical customer count based on annual growth for 2020-2021

customer_data = (
    philippine_sales[
        philippine_sales["Year"].isin([2024, 2025])
        & philippine_sales["Customer Class"].isin(
            ["Residential", "Commercial"]
        )
    ]
    .groupby(["Year", "Customer Class"])["Customer Count"]
    .sum()
    .reset_index()
)

customer_growth = {}

for customer_class in ["Residential", "Commercial"]:

    data = (
        customer_data[
            customer_data["Customer Class"] == customer_class
        ]
        .set_index("Year")
    )

    growth = (
        data.loc[2025, "Customer Count"]
        / data.loc[2024, "Customer Count"]
    ) - 1

    customer_growth[customer_class] = growth

hypothetical_customers = []

for customer_class in ["Residential", "Commercial"]:

    data = (
        customer_data[
            customer_data["Customer Class"] == customer_class
        ]
        .set_index("Year")
    )

    growth = customer_growth[customer_class]

    customers_2024 = data.loc[2024, "Customer Count"]

    # Back-calculate from 2024
    customers_2021 = (
        customers_2024 / (1 + growth) ** 3
    )

    customers_2020 = (
        customers_2024 / (1 + growth) ** 4
    )

    hypothetical_customers.extend([
        {
            "Year": 2020,
            "Customer Class": customer_class,
            "Customer Count": customers_2020,
            "Data Type": "Hypothetical"
        },
        {
            "Year": 2021,
            "Customer Class": customer_class,
            "Customer Count": customers_2021,
            "Data Type": "Hypothetical"
        },
        {
            "Year": 2024,
            "Customer Class": customer_class,
            "Customer Count": data.loc[
                2024, "Customer Count"
            ],
            "Data Type": "Actual"
        },
        {
            "Year": 2025,
            "Customer Class": customer_class,
            "Customer Count": data.loc[
                2025, "Customer Count"
            ],
            "Data Type": "Actual"
        }
    ])


hypothetical_customers = pd.DataFrame(
    hypothetical_customers
)


#print("\nCustomer count assumptions:")
#print(
#    hypothetical_customers
#    .round(0)
#    .to_string(index=False)
#)

philippine_sales_model = philippine_sales.copy()

philippine_sales_model.loc[
    philippine_sales_model["Year"].isin([2020, 2021]),
    "Customer Count"
] = np.nan

philippine_sales_model = philippine_sales_model.merge(
    hypothetical_customers[
        ["Year", "Customer Class", "Customer Count", "Data Type"]
    ],
    on=["Year", "Customer Class"],
    how="left",
    suffixes=("", "_Model")
)

philippine_sales_model["Customer Count"] = (
    philippine_sales_model["Customer Count_Model"]
)

philippine_sales_model.drop(
    columns=["Customer Count_Model"],
    inplace=True
)


print(
    philippine_sales_model[
        [
            "Year",
            "Month",
            "Customer Class",
            "Sales",
            "Customer Count",
            "Data Type"
        ]
    ].head(20)
)

In [ ]:
sales = philippine_sales_model[
    philippine_sales_model["Customer Class"].isin(CUSTOMER_CLASSES)
].copy()

sales = sales[
    sales["Year"].isin(WFH_YEARS + OFFICE_YEARS)
].copy()

#print(sales)

def assign_work_mode(row):

    if (
        row["Year"] in WFH_YEARS
        and row["Customer Class"] == "Residential"
    ):
        return "WFH"

    elif (
        row["Year"] in OFFICE_YEARS
        and row["Customer Class"] == "Commercial"
    ):
        return "Office"

    return np.nan


sales["Work_Mode"] = (
    sales.apply(
        assign_work_mode,
        axis=1
    )
)



comparison_sales = sales[
    sales
    ["Work_Mode"].notna()
].copy()

print(comparison_sales)

In [ ]:
annual_sales = (
    comparison_sales
    .groupby(
        [
            "Year",
            "Customer Class",
            "Work_Mode"
        ],
        as_index=False
    )
    .agg(
        Annual_Sales_MWh=("Sales", "sum"),
        Average_Customers=("Customer Count", "mean")
    )
)

# Identify whether customer counts are actual or back-cast.
data_type_lookup = (
    hypothetical_customers
    [["Year", "Customer Class", "Data Type"]]
    .drop_duplicates()
)

annual_sales = annual_sales.merge(
    data_type_lookup,
    on=["Year", "Customer Class"],
    how="left"
)

annual_sales["MWh_per_Customer"] = (
    annual_sales["Annual_Sales_MWh"]
    / annual_sales["Average_Customers"]
)

print(annual_sales.round(4).to_string(index=False))

In [ ]:
rates = rates_luzon.copy()

rates["Date"] = pd.to_datetime(
    rates["Date"],
    errors="coerce"
)

rates["Year"] = rates["Date"].dt.year

rates = rates[
    rates["Year"].isin(WFH_YEARS + OFFICE_YEARS)
].copy()

# Keep the rate data as observed. Missing rates are left as NaN
# rather than replacing them with an overall mean.
annual_rates = (
    rates
    .dropna(subset=["Rate"])
    .groupby(
        ["Year", "Customer Class"],
        as_index=False
    )
    .agg(
        Average_Rate_PHP_kWh=("Rate", "mean")
    )
)

print("Annual electricity rates:")
print(annual_rates.round(4).to_string(index=False))

In [ ]:
annual_model = annual_sales.merge(
    annual_rates,
    on=["Year", "Customer Class"],
    how="left"
)

annual_model[
    "Estimated_Electricity_Cost_PHP"
] = (
    annual_model["Annual_Sales_MWh"]
    * 1000
    * annual_model["Average_Rate_PHP_kWh"]
)


# Electricity cost per customer
annual_model[
    "Cost_per_Customer_PHP"
] = (
    annual_model[
        "Estimated_Electricity_Cost_PHP"
    ]
    / annual_model["Average_Customers"]
)


print(annual_model) #

In [ ]:
energy = energy_mix_final.copy()

energy = energy[
    energy["Year"].isin(WFH_YEARS + OFFICE_YEARS)
].copy()

energy_pivot = (
    energy
    .pivot_table(
        index="Year",
        columns="Energy Source",
        values="Share",
        aggfunc="sum"
    )
    .reset_index()
)


energy_pivot = energy_pivot.rename(
    columns={
        "Renewable Energy (RE)": "Renewable_Share",
        "Coal": "Coal_Share",
        "Natural Gas": "Natural_Gas_Share",
        "Oil-based": "Oil_Share"
    }
)


for col in [
    "Coal_Share",
    "Natural_Gas_Share",
    "Oil_Share",
    "Renewable_Share"
]:

    if col not in energy_pivot.columns:
        energy_pivot[col] = 0


energy_pivot["Fossil_Share"] = (
    energy_pivot["Coal_Share"]
    + energy_pivot["Natural_Gas_Share"]
    + energy_pivot["Oil_Share"]
)


annual_model = annual_model.merge(
    energy_pivot,
    on="Year",
    how="left"
)

In [ ]:
yearly_results = annual_model[
    [
        "Year",
        "Work_Mode",
        "Customer Class",
        "Annual_Sales_MWh",
        "Average_Customers",
        "MWh_per_Customer",
        "Average_Rate_PHP_kWh",
        "Estimated_Electricity_Cost_PHP",
        "Cost_per_Customer_PHP",
        "Renewable_Share",
        "Fossil_Share"
    ]
].sort_values("Year")

print("YEAR-BY-YEAR COMPARISON")
print("==============================================")

print(
    yearly_results.round(4)
    .to_string(index=False)
)


In [ ]:

wfh_data = annual_model[
    annual_model["Work_Mode"] == "WFH"
]

office_data = annual_model[
    annual_model["Work_Mode"] == "Office"
]


# Average annual electricity intensity
wfh_mwh_customer = (
    wfh_data["MWh_per_Customer"].mean()
)

office_mwh_customer = (
    office_data["MWh_per_Customer"].mean()
)


# Average electricity cost per customer
wfh_cost_customer = (
    wfh_data["Cost_per_Customer_PHP"].mean()
)

office_cost_customer = (
    office_data["Cost_per_Customer_PHP"].mean()
)


# Average electricity rate
wfh_rate = (
    wfh_data["Average_Rate_PHP_kWh"].mean()
)

office_rate = (
    office_data["Average_Rate_PHP_kWh"].mean()
)


# Total electricity sales
wfh_total_sales = (
    wfh_data["Annual_Sales_MWh"].sum()
)

office_total_sales = (
    office_data["Annual_Sales_MWh"].sum()
)

wfh_peak_demand = hourly_demand.loc[
    hourly_demand["Year"].isin(WFH_YEARS),
    "Demand_MW"
].max()

office_peak_demand = hourly_demand.loc[
    hourly_demand["Year"].isin(OFFICE_YEARS),
    "Demand_MW"
].max()

peak_change = (
    (office_peak_demand / wfh_peak_demand) - 1
) * 100

In [ ]:
comparison = pd.DataFrame({

    "Metric": [
        "Total Electricity Sales (MWh)",
        "Average MWh per Customer",
        "Average Electricity Rate (PHP/kWh)",
        "Average Electricity Cost per Customer (PHP)",
        "Peak Demand (MW)"
    ],

    "WFH_2020_2021": [
        wfh_total_sales,
        wfh_mwh_customer,
        wfh_rate,
        wfh_cost_customer,
        wfh_peak_demand
    ],

    "Office_2024_2025": [
        office_total_sales,
        office_mwh_customer,
        office_rate,
        office_cost_customer,
        office_peak_demand
    ]
})


comparison[
    "Office_vs_WFH_%_Change"
] = (
    comparison["Office_2024_2025"]
    / comparison["WFH_2020_2021"]
    - 1
) * 100


print("WFH VS OFFICE COMPARISON")
print("==============================================")

print(
    comparison.round(2)
    .to_string(index=False)
)


In [ ]:
energy_index = (
    office_mwh_customer
    / wfh_mwh_customer
) * 100


energy_index_table = pd.DataFrame({

    "Work_Mode": [
        "WFH",
        "Office"
    ],

    "Energy_Intensity_Index": [
        100,
        energy_index
    ]
})

cost_index = (
    office_cost_customer
    / wfh_cost_customer
) * 100


cost_index_table = pd.DataFrame({

    "Work_Mode": [
        "WFH",
        "Office"
    ],

    "Electricity_Cost_Index": [
        100,
        cost_index
    ]
})

ENERGY_WEIGHT = 0.50
COST_WEIGHT = 0.50


burden_index = pd.DataFrame({

    "Work_Mode": [
        "WFH",
        "Office"
    ],

    "Energy_Index": [
        100,
        energy_index
    ],

    "Cost_Index": [
        100,
        cost_index
    ]
})


burden_index[
    "Electricity_Burden_Index"
] = (
    ENERGY_WEIGHT
    * burden_index["Energy_Index"]
    +
    COST_WEIGHT
    * burden_index["Cost_Index"]
)


print("\n==============================================")
print("ELECTRICITY BURDEN INDEX")
print("==============================================")

print(
    burden_index.round(2)
    .to_string(index=False)
)


In [ ]:
energy_mix_comparison = (
    energy_mix_final
    .pivot_table(
        index="Energy Source",
        columns="Year",
        values="Share",
        aggfunc="sum"
    )
    .reset_index()
)


print("\n==============================================")
print("LUZON ENERGY MIX")
print("==============================================")

print(
    energy_mix_comparison.round(2)
    .to_string(index=False)
)

<h2>Results: WFH vs Office Electricity Model</h2>


In [ ]:
print("==============================================")
print("KEY MODEL RESULTS")
print("==============================================")

print(f"WFH average MWh/customer: {wfh_mwh_customer:.6f}")
print(f"Office average MWh/customer: {office_mwh_customer:.6f}")
print(f"Office vs WFH energy intensity: {energy_index:.2f}")
print(f"WFH average cost/customer: PHP {wfh_cost_customer:,.2f}")
print(f"Office average cost/customer: PHP {office_cost_customer:,.2f}")
print(f"Office vs WFH cost index: {cost_index:.2f}")
print("WFH Electricity Burden Index: 100.00")
print("Office Electricity Burden Index: "
    f"{burden_index.loc[burden_index['Work_Mode'] == 'Office', 'Electricity_Burden_Index'].iloc[0]:.2f}")

In [ ]:
plot_data = annual_model.copy()

plt.figure(figsize=(10, 6))

for mode in ["WFH", "Office"]:

    data = plot_data[
        plot_data["Work_Mode"] == mode
    ]

    plt.plot(
        data["Year"],
        data["MWh_per_Customer"],
        marker="o",
        label=mode
    )

plt.xlabel("Year")
plt.ylabel("MWh per Customer")
plt.title(
    "Electricity Consumption per Customer: WFH vs Office"
)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 6))

for mode in ["WFH", "Office"]:

    data = plot_data[
        plot_data["Work_Mode"] == mode
    ]

    plt.plot(
        data["Year"],
        data["Cost_per_Customer_PHP"],
        marker="o",
        label=mode
    )

plt.xlabel("Year")
plt.ylabel("PHP per Customer")
plt.title(
    "Electricity Cost per Customer: WFH vs Office"
)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()



In [ ]:
plt.figure(figsize=(8, 5))

plt.bar(
    burden_index["Work_Mode"],
    burden_index["Electricity_Burden_Index"]
)

plt.axhline(
    100,
    linestyle="--",
    linewidth=1
)

plt.ylabel("Electricity Burden Index")
plt.title(
    "Electricity Burden Index\n(WFH = 100)"
)

plt.tight_layout()
plt.show()


In [ ]:
energy_plot = energy_mix_comparison.set_index(
    "Energy Source"
).T

energy_plot.plot(
    kind="bar",
    stacked=True,
    figsize=(10, 6)
)

plt.xlabel("Year")
plt.ylabel("Share (%)")
plt.title("Luzon Electricity Generation Mix")
plt.legend(title="Energy Source")
plt.tight_layout()
plt.show()

In [ ]:
hourly_plot = (
    hourly_demand
    .groupby(["Work_Mode", "Hour"], as_index=False)["Demand_MW"]
    .mean()
)

hourly_plot_pivot = (
    hourly_plot
    .pivot(index="Hour", columns="Work_Mode", values="Demand_MW")
)

plt.figure(figsize=(12, 6))

if "WFH" in hourly_plot_pivot.columns:
    plt.plot(
        hourly_plot_pivot.index,
        hourly_plot_pivot["WFH"],
        marker="o",
        label="WFH (2020–2021)"
    )

if "Office" in hourly_plot_pivot.columns:
    plt.plot(
        hourly_plot_pivot.index,
        hourly_plot_pivot["Office"],
        marker="o",
        label="Office (2024–2025)"
    )

plt.title("Average Hourly Electricity Demand: WFH vs Office")
plt.xlabel("Hour of Day")
plt.ylabel("Average Demand (MW)")
plt.xticks(range(1, 25))
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## Cost-Efficiency Result


In [ ]:
if office_cost_customer < wfh_cost_customer:
    lower_cost_scenario = "Office"
elif wfh_cost_customer < office_cost_customer:
    lower_cost_scenario = "WFH"
else:
    lower_cost_scenario = "Equal"

cost_difference = abs(office_cost_customer - wfh_cost_customer)

cost_change_percent = (
    (office_cost_customer / wfh_cost_customer) - 1
) * 100

print("MODEL-BASED COST RESULT")
print("==============================================")
print(f"WFH average cost/customer: PHP {wfh_cost_customer:,.2f}")
print(f"Office average cost/customer: PHP {office_cost_customer:,.2f}")
print(f"Absolute cost difference/customer: PHP {cost_difference:,.2f}")
print(f"Office vs WFH cost change: {cost_change_percent:+.2f}%")
print(f"Lower modeled electricity-cost scenario: {lower_cost_scenario}")
print(f"WFH peak demand: {wfh_peak_demand:,.2f} MW")
print(f"Office peak demand: {office_peak_demand:,.2f} MW")
print(f"Office vs WFH peak-demand change: {peak_change:+.2f}%")

## Monte Carlo Simulations for Predictions


### Peak Demand Simulation


In [ ]:
iterations = 100 #years
days_in_year = 365

# isolate the daily peak demands from the hourly dataset
daily_peaks = hourly_demand.groupby(['Year', 'Month', 'Day'])['Demand_MW'].max().reset_index()

#WFH and Office historical daily peaks
wfh_peaks = daily_peaks[daily_peaks['Year'].isin(WFH_YEARS)]['Demand_MW'].dropna()
office_peaks = daily_peaks[daily_peaks['Year'].isin(OFFICE_YEARS)]['Demand_MW'].dropna()

#fit exact non-normal distribution shapes with KDE
wfh_kde = stats.gaussian_kde(wfh_peaks)
office_kde = stats.gaussian_kde(office_peaks)

#10,000 scenarios resampling
sim_wfh_daily = wfh_kde.resample(iterations * days_in_year).reshape(iterations, days_in_year)
sim_office_daily = office_kde.resample(iterations * days_in_year).reshape(iterations, days_in_year)

# worst day of each year
sim_wfh_annual_peaks = np.max(sim_wfh_daily, axis=1)
sim_office_annual_peaks = np.max(sim_office_daily, axis=1)

#
wfh_median_peak = np.median(sim_wfh_annual_peaks)
office_median_peak = np.median(sim_office_annual_peaks)
wfh_95th_peak = np.percentile(sim_wfh_annual_peaks, 95)
office_95th_peak = np.percentile(sim_office_annual_peaks, 95)

print("KDE Peak Demand Forecast for 100 Years")
print(f"WFH Expected Annual Peak (Median): {wfh_median_peak:,.2f} MW")
print(f"WFH Extreme Grid Strain (95th): {wfh_95th_peak:,.2f} MW\n")
print(f"Office Expected Annual Peak (Median): {office_median_peak:,.2f} MW")
print(f"Office Extreme Grid Strain (95th): {office_95th_peak:,.2f} MW\n")

plt.figure(figsize=(10, 6))
plt.hist(sim_wfh_annual_peaks, bins=50, alpha=0.7, label='WFH Peak Demand')
plt.hist(sim_office_annual_peaks, bins=50, alpha=0.7, label='Office Peak Demand')
plt.axvline(wfh_95th_peak, color='blue', linestyle='dashed', linewidth=2, label='WFH 95th Risk Threshold')
plt.axvline(office_95th_peak, color='orange', linestyle='dashed', linewidth=2, label='Office 95th Risk Threshold')
plt.title('Forecasted Annual Peak Demand (WFH vs Office)')
plt.xlabel('Annual Peak Demand (MW)')
plt.ylabel('Frequency (Probability within 100 years)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Cost-Efficiency Simulation


In [ ]:
months_in_year = 12
# baseline from sales and rates
wfh_sales = comparison_sales[comparison_sales['Work_Mode'] == 'WFH'].copy()
office_sales = comparison_sales[comparison_sales['Work_Mode'] == 'Office'].copy()

#monthly kwh per customer
wfh_kwh_data = ((wfh_sales['Sales'] / wfh_sales['Customer Count']) * 1000).dropna()
office_kwh_data = ((office_sales['Sales'] / office_sales['Customer Count']) * 1000).dropna()
wfh_rates_data = rates_luzon[(rates_luzon['Customer Class'] == 'Residential') & (rates_luzon['Date'].dt.year.isin(WFH_YEARS))]['Rate'].dropna()
office_rates_data = rates_luzon[(rates_luzon['Customer Class'] == 'Commercial') & (rates_luzon['Date'].dt.year.isin(OFFICE_YEARS))]['Rate'].dropna()

#fit exact non-normal distribution shapes with KDE
wfh_kwh_kde = stats.gaussian_kde(wfh_kwh_data)
office_kwh_kde = stats.gaussian_kde(office_kwh_data)
wfh_rate_kde = stats.gaussian_kde(wfh_rates_data)
office_rate_kde = stats.gaussian_kde(office_rates_data)

# resampling
sim_wfh_kwh = wfh_kwh_kde.resample(iterations * months_in_year).reshape(iterations, months_in_year)
sim_office_kwh = office_kwh_kde.resample(iterations * months_in_year).reshape(iterations, months_in_year)
sim_wfh_rates = wfh_rate_kde.resample(iterations * months_in_year).reshape(iterations, months_in_year)
sim_office_rates = office_rate_kde.resample(iterations * months_in_year).reshape(iterations, months_in_year)

# annual cost
sim_wfh_annual_cost = np.sum(sim_wfh_kwh * sim_wfh_rates, axis=1)
sim_office_annual_cost = np.sum(sim_office_kwh * sim_office_rates, axis=1)

#
wfh_median_cost = np.median(sim_wfh_annual_cost)
office_median_cost = np.median(sim_office_annual_cost)
wfh_95th_cost = np.percentile(sim_wfh_annual_cost, 95)
office_95th_cost = np.percentile(sim_office_annual_cost, 95)

print("KDE Forecasted Annual Cost Per Customer For 100 Years")
print(f"WFH Expected Annual Cost (Median): PHP {wfh_median_cost:,.2f}")
print(f"WFH Worst-Case Cost Budget (95th): PHP {wfh_95th_cost:,.2f}\n")
print(f"Office Expected Annual Cost (Median): PHP {office_median_cost:,.2f}")
print(f"Office Worst-Case Cost Budget (95th): PHP {office_95th_cost:,.2f}\n")

plt.figure(figsize=(10, 6))
plt.hist(sim_wfh_annual_cost, bins=50, alpha=0.7, label='WFH Annual Cost')
plt.hist(sim_office_annual_cost, bins=50, alpha=0.7, label='Office Annual Cost')
plt.axvline(wfh_95th_cost, color='blue', linestyle='dashed', linewidth=2, label='WFH 95th Risk Threshold')
plt.axvline(office_95th_cost, color='orange', linestyle='dashed', linewidth=2, label='Office 95th Risk Threshold')
plt.title('Forecasted Annual Electricity Cost per Customer (WFH vs Office)')
plt.xlabel('Simulated Annual Cost (PHP)')
plt.ylabel('Frequency (Probability within 100 years)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()